In [10]:
import os
import numpy as np
import pandas as pd
import scipy.io
from tqdm import tqdm
from scipy.signal import butter, sosfiltfilt, decimate

In [11]:
raw_data_root = r"D:\M143020071\MACE\raw_data_result\conversion_data"
healthy_dir = os.path.join(raw_data_root, "healthy")
patient_dir = os.path.join(raw_data_root, "patient")

# 儲存路徑
save_root = r"D:\M143020071\MACE and MI\raw_data_result\iSKNA_signal\ch1\sr500_0.5_50_MI_win10s_step2s_2-7m"
non_MACE_save_dir = os.path.join(save_root, "non_MACE")
MACE_save_dir = os.path.join(save_root, "MACE")

for d in [non_MACE_save_dir, MACE_save_dir]:
    if not os.path.exists(d):
        os.makedirs(d)
        print(f"已建立資料夾: {d}")

In [12]:
def butter_bandpass_sos(lowcut, highcut, fs, order=6):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    sos = butter(order, [low, high], btype='band', output='sos')
    return sos

def butter_lowpass_sos(cutoff, fs, order=6):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    sos = butter(order, normal_cutoff, btype='low', output='sos')
    return sos


def butter_highpass_sos(cutoff, fs, order=6):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    sos = butter(order, normal_cutoff, btype='high', output='sos')
    return sos

def sliding_window_extract(signal, fs=10, window_sec=10, step_sec=2):
    """
    對訊號進行滑動視窗切割，並保留完整的訊號片段作為特徵。
    預期輸入 signal 為降頻後 (10Hz) 的一維陣列。
    """
    window_size = window_sec * fs  # 60 * 10 = 600 點
    step_size = step_sec * fs      # 1 * 10 = 10 點
    
    windows = []
    

    for start in range(0, len(signal) - window_size + 1, step_size):
        end = start + window_size
        window_data = signal[start:end] 
        windows.append(window_data)
        

    return np.array(windows)


fs = 10000 
two_min_pts = 2 * fs * 60
five_min_pts = 5 * fs * 60
seven_min_pts = 7 * fs * 60
downsample=20
target_fs = fs // downsample

In [13]:
tasks = []
# 定義要排除的非資料檔名
exclude_list = ['error_subjects.csv', 'only_signal.csv']

# 收集 healthy 檔案 (Label 0)
if os.path.exists(healthy_dir):
    for f in os.listdir(healthy_dir):
        # 條件：是 .csv 檔 且 不在排除名單內 且 檔名第一個字是數字 (代表是受試者 ID)
        if f.endswith('.csv') and f not in exclude_list and f[0].isdigit():
            tasks.append({'path': os.path.join(healthy_dir, f), 'label': 0, 'name': f})

# 收集 patient 檔案 (Label 1)
if os.path.exists(patient_dir):
    for f in os.listdir(patient_dir):
        # 同樣的過濾邏輯
        if f.endswith('.csv') and f not in exclude_list and f[0].isdigit():
            tasks.append({'path': os.path.join(patient_dir, f), 'label': 1, 'name': f})

print(f"過濾後的處理總人數: {len(tasks)} ")

過濾後的處理總人數: 522 


In [14]:
processed_count = 0

for task in tqdm(tasks, desc="Processing CSV Data"):
    file_path = task['path']
    label = task['label']
    file_name = task['name']
    
    try:
        # A. 讀取 CSV 並抓取 Ch1 (第二欄)
        # 注意: 如果 CSV 有 Header，header=0；若無則 header=None
        df_temp = pd.read_csv(file_path)
        signal = df_temp.iloc[:, 1].values # 取得第二欄數值
        
        # B. 濾波流程 (維持 0.5-50 Hz)
        sos_high = butter_highpass_sos(0.5, fs, order=6)
        temp_sig = sosfiltfilt(sos_high, signal)
        sos_low = butter_lowpass_sos(50, fs, order=6)
        filtered_sig = sosfiltfilt(sos_low, temp_sig)
        
        # C. 降頻
        
        iskna_downsampled = decimate(filtered_sig, downsample, ftype='iir', zero_phase=True)
        
        # D. 擷取時段 (2-7 分鐘)
        # 降頻後的頻率為 500Hz
        current_len = len(iskna_downsampled)
        min_len_2min = 2 * target_fs * 60
        max_len_7min = 7 * target_fs * 60
        
        if current_len >= 2 * 500 * 60:
            if current_len < 7 * 500 * 60:
                # 抓最後 5 分鐘
                segment = iskna_downsampled[-5 * 500 * 60:]
            else:
                # 抓 2-7 分鐘這區間
                segment = iskna_downsampled[2 * 500 * 60 : 7 * 500 * 60]
        else:
            print(f"檔案 {file_name} 長度不足 2 分鐘，跳過。")
            continue
        # E. 滑動視窗 (10s window, 2s step)
        windowed_signal = sliding_window_extract(segment, fs=target_fs, window_sec=10, step_sec=2)
        
        # F. 正規化與合併標籤
        window_means = np.mean(windowed_signal, axis=1, keepdims=True)
        window_stds  = np.std(windowed_signal, axis=1, keepdims=True)
        windowed_signal_nor = (windowed_signal - window_means) / (window_stds + 1e-8)
        
        internal_label = 1
        label_arr = np.full((windowed_signal.shape[0], 1), internal_label)
        final_save_data = np.hstack([label_arr, windowed_signal_nor]).astype(np.float32)
        
        # G. 儲存 NPY
        target_dir = non_MACE_save_dir if label == 0 else MACE_save_dir
        save_file_path = os.path.join(target_dir, file_name.replace('.csv', '.npy'))
        np.save(save_file_path, final_save_data)
        
        processed_count += 1
        
    except Exception as e:
        print(f"處理 {file_name} 時出錯: {e}")

print(f"\n所有程序完成！共成功處理並儲存 {processed_count} 位受試者的資料。")

Processing CSV Data: 100%|██████████| 522/522 [23:46<00:00,  2.73s/it]


所有程序完成！共成功處理並儲存 522 位受試者的資料。
